# Digital Image Processing Project
## Fruit Sorting System by Color and Size

**Objective**: Create an industrial pipeline to sort fruits using advanced image processing techniques.

---

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Tuple
from google.colab import drive

# Constants for tuning
GAUSSIAN_KERNEL = (5, 5)
MEDIAN_KSIZE = 5
FIGURE_SIZE = (12, 6)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Section 1: Pre-processing
การปรับปรุงคุณภาพภาพเพื่อลด Noise และเพิ่ม Contrast ให้วัตถุแยกออกจากพื้นหลังได้ชัดเจนที่สุด

### 1.1 Helper Functions (Visualization)
สร้างฟังก์ชันสำหรับการแสดงผลภาพเปรียบเทียบแบบ Side-by-Side

In [2]:
def show_comparison(img1: np.ndarray, img2: np.ndarray, title1: str = "Original", title2: str = "Processed") -> None:
    """Displays two images side by side for easy comparison."""
    plt.figure(figsize=FIGURE_SIZE)

    # Left image
    plt.subplot(1, 2, 1)
    if len(img1.shape) == 3:
        plt.imshow(cv2.cvtColor(img1, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img1, cmap='gray')
    plt.title(title1)
    plt.axis('off')

    # Right image
    plt.subplot(1, 2, 2)
    if len(img2.shape) == 3:
        plt.imshow(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img2, cmap='gray')
    plt.title(title2)
    plt.axis('off')

    plt.tight_layout()
    plt.show()

### 1.2 Image Acquisition

In [4]:
def load_image(path: str) -> Optional[np.ndarray]:
    """Loads an image with basic error handling."""
    try:
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(f"Cannot find image at {path}")
        return img
    except Exception as e:
        print(f"Error: {e}")
        return None

### 1.3 Histogram Equalization (Manual Implementation)
การใช้ CDF เพื่อกระจายความเข้มแสงให้ทั่วภาพ ช่วยให้รายละเอียดในส่วนที่มืดหรือสว่างเกินไปชัดเจนขึ้น

In [5]:
def manual_histogram_equalization(gray_img: np.ndarray) -> np.ndarray:
    """Manually implements global histogram equalization."""
    # Ensure image is grayscale
    if len(gray_img.shape) != 2:
        gray_img = cv2.cvtColor(gray_img, cv2.COLOR_BGR2GRAY)

    # 1. Histogram Calculation
    hist, _ = np.histogram(gray_img.flatten(), 256, [0, 256])

    # 2. CDF Calculation
    cdf = hist.cumsum()

    # 3. CDF Normalization (Formula based on DIP Theory)
    cdf_m = np.ma.masked_equal(cdf, 0)
    cdf_m = (cdf_m - cdf_m.min()) * 255 / (cdf_m.max() - cdf_m.min())
    equalized_lookup_table = np.ma.filled(cdf_m, 0).astype('uint8')

    # 4. Image Mapping
    return equalized_lookup_table[gray_img]

### 1.4 Noise Reduction Filters

In [6]:
def denoise_image(img: np.ndarray, method: str = 'gaussian') -> np.ndarray:
    """
    Reduces noise using specified method.
    'gaussian': Smooths general electronic noise.
    'median': Best for salt-and-pepper noise.
    """
    if method.lower() == 'gaussian':
        return cv2.GaussianBlur(img, GAUSSIAN_KERNEL, 0)
    elif method.lower() == 'median':
        return cv2.medianBlur(img, MEDIAN_KSIZE)
    else:
        return img

In [ ]:
# ใส่ตำแหน่งที่อยู่ของไฟล์ภาพใน Google Drive ของคุณ
image_path = ''

# 1. ทดสอบการโหลดภาพ
original_img = load_image(image_path)

if original_img is not None:
    # 2. ทดลองลด Noise ทำ Gaussian Blur
    denoised_img = denoise_image(original_img, method='gaussian')
    show_comparison(original_img, denoised_img, "Original Image", "After Gaussian Blur")

    # 3. ทดลองปรับความสว่าง (Histogram Equalization)
    gray_img = cv2.cvtColor(denoised_img, cv2.COLOR_BGR2GRAY)
    eq_img = manual_histogram_equalization(gray_img)
    show_comparison(gray_img, eq_img, "Grayscale Image", "Manual Histogram Equalization")


## Section 2: Step 2 — Color Segmentation
การแปลง Color Space (BGR ไปเป็น HSV) เพื่อให้ง่ายต่อการตรวจจับสีผลไม้ และใช้ Mask หรือเทคนิค Otsu เพื่อตีกรอบวัตถุ

In [ ]:
def create_color_mask(bgr_img: np.ndarray, color_type: str = 'red') -> np.ndarray:
    """Converts BGR to HSV and returns a binary mask for the specified color."""
    hsv_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2HSV)
    
    if color_type.lower() == 'red':
        mask1 = cv2.inRange(hsv_img, np.array([0, 100, 100]), np.array([10, 255, 255]))
        mask2 = cv2.inRange(hsv_img, np.array([170, 100, 100]), np.array([180, 255, 255]))
        return cv2.bitwise_or(mask1, mask2)
        
    elif color_type.lower() == 'yellow':
        return cv2.inRange(hsv_img, np.array([15, 100, 100]), np.array([35, 255, 255]))
        
    elif color_type.lower() == 'green':
        return cv2.inRange(hsv_img, np.array([36, 50, 50]), np.array([85, 255, 255]))
        
    return np.zeros(hsv_img.shape[:2], dtype=np.uint8)

In [ ]:
def otsu_segmentation(bgr_img: np.ndarray) -> np.ndarray:
    """Uses Otsu's thresholding to segment object from background."""
    gray_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray_img, GAUSSIAN_KERNEL, 0)
    _, binary_mask = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Invert mask if background is white 
    inverted_mask = cv2.bitwise_not(binary_mask)
    return inverted_mask

In [ ]:
if original_img is not None:
    print("--- Testing Segmentation Algorithms ---")
    
    otsu_mask = otsu_segmentation(original_img)
    show_comparison(original_img, otsu_mask, "Original Image", "Otsu's Binarization")
    
    mask_red = create_color_mask(original_img, 'red')
    mask_yellow = create_color_mask(original_img, 'yellow')
    mask_green = create_color_mask(original_img, 'green')
    
    show_comparison(original_img, mask_green, "Original", "Green Mask (e.g. Grapes)")
    show_comparison(mask_red, mask_yellow, "Red Mask", "Yellow Mask (e.g. Mango/Banana)")
